# 🐾 Deep Learning in the Kalahari · Session 1 of 3
## *Understanding Deep Learning*, chapters 1–4: models

**Transition Bio · ML Reading Group**

This series works through chapters 1–8 of Simon J.D. Prince's **_Understanding Deep Learning_** (MIT Press; free at [udlbook.github.io](https://udlbook.github.io/udlbook/)) in three sessions. Each one is about **20 min of slides and 60–70 min of hands-on puzzles**. We follow the book's notation, square brackets and all: $\mathbf{y} = \mathbf{f}[\mathbf{x}, \boldsymbol\phi]$.

| Session | Topic | Chapters |
|---|---|---|
| **Session 1** | Models | Ch 1–4 |
| Session 2 | Losses and training | Ch 5–6 |
| Session 3 | Backprop and performance | Ch 7–8 |

---

### 🦦 The story
You've just joined a long-running field station in the Kalahari that studies a wild meerkat mob. The mob is about thirty animals, run by a dominant female (in meerkat society, she's in charge). The station has a lot of data and no deep-learning expert, so you're it. Every chapter sets you a new problem from the field.

All datasets are **synthetic**. They're generated in this notebook so that we know the ground truth and can check your answers.

### 🧭 How each part works
| Symbol | Meaning |
|---|---|
| 📖 | Recap of the chapter's maths, with the equations derived properly |
| 💻 | We build it **from scratch in NumPy** first; PyTorch only appears once you've earned it |
| 🧩 ⭐ / ⭐⭐ / ⭐⭐⭐ | Puzzle: quick / medium / stretch. Fill in the `TODO`, then run the 🧪 check cell |
| 🔑 | Hidden solution cell. Double-click to open it. Have a go first; it's there when you need it |
| 🏆 | Open challenge with bronze / silver / gold targets to aim for |

> **Important:** later cells use the functions you write. If you get stuck, **run the 🔑 solution cell** (you don't have to read it) so the rest of the notebook still works.

### 🗺️ This session (≈ 70 min hands-on)
| | | Suggested time |
|---|---|---|
| Part 1 | **Introduction** (Ch 1): supervised, unsupervised and reinforcement learning | ~10 min |
| Part 2 | **Supervised learning** (Ch 2): the Great Pup Weigh-in | ~15 min |
| Part 3 | **Shallow neural networks** (Ch 3): digging burrows with ReLUs | ~25 min |
| Part 4 | **Deep neural networks** (Ch 4): folding the dune sea | ~20 min |

⏱ **Short on time?** Skip the stretch puzzles **2.3, 3.4, 4.4** in the room and finish them afterwards; they're good practice to finish afterwards.

At the end, `progress()` shows which puzzles you've solved and which are left. It's just for you; nothing is recorded or shared.

*Book figures and problem numbers are cited as e.g. "Fig 3.3" and "P3.8".*

In [ ]:
#@title 🐾 Setup: run me first! (plotting style, score keeper, referee)
import math, itertools, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
warnings.filterwarnings("ignore", category=RuntimeWarning)
np.set_printoptions(precision=4, suppress=True)

# Transition Bio palette
TB = dict(green="#008759", lgreen="#73AE63", blue="#1366AB", lblue="#73B0E6",
          purple="#8F64B7", red="#CF1D10", orange="#EF8E00", grey="#6B6B6B", sand="#E9D8B4")
plt.rcParams.update({
    "figure.dpi": 110, "figure.figsize": (6, 4), "axes.spines.top": False,
    "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25,
    "axes.prop_cycle": plt.cycler(color=[TB["green"], TB["blue"], TB["orange"],
                                         TB["purple"], TB["red"], TB["lblue"], TB["lgreen"]]),
    "font.size": 10.5, "axes.titleweight": "bold"})

# ---------------- progress tracker (private to you; nothing is shared) ----------------
_SOLVED, _PEEKED, _SEEN = set(), set(), set()
SESSION = 1

def peek(pid):
    _PEEKED.add(pid)

def check(pid, stars, test, ok_msg="", hint=""):
    """Run `test` (a zero-arg function returning True/False) and report the result."""
    _SEEN.add(pid)
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"🕳️  Puzzle {pid}: still a TODO. Fill it in and re-run (or run the 🔑 cell).")
        return
    except Exception as e:
        print(f"💥 Puzzle {pid} raised {type(e).__name__}: {e}")
        if hint: print("   💡 Hint:", hint)
        return
    if ok:
        _SOLVED.add(pid)
        print(f"✅ Puzzle {pid} solved!  {ok_msg}")
    else:
        print(f"❌ Puzzle {pid}: not quite.", ("💡 " + hint) if hint else "")

def progress():
    own = sorted(_SOLVED - _PEEKED); helped = sorted(_SOLVED & _PEEKED); todo = sorted(_SEEN - _SOLVED)
    print(f"🐾 {len(_SOLVED)} of {len(_SEEN)} puzzles solved "
          f"({len(own)} on your own, {len(helped)} after opening the solution).")
    if todo: print("   Still to do:", ", ".join(todo))

def close(a, b, tol=1e-6):
    a, b = np.asarray(a, float), np.asarray(b, float)
    return a.shape == b.shape and np.allclose(a, b, atol=tol, rtol=tol)

def num_grad(fn, params, eps=1e-6):
    """Central finite-difference gradient of scalar fn(params) w.r.t. a 1D array."""
    params = np.array(params, float); g = np.zeros_like(params)
    for i in range(len(params)):
        d = np.zeros_like(params); d[i] = eps
        g[i] = (fn(params + d) - fn(params - d)) / (2 * eps)
    return g

def medal(score, bronze, silver, gold, higher_is_better=False):
    s = score if higher_is_better else -score
    b, si, g = (bronze, silver, gold) if higher_is_better else (-bronze, -silver, -gold)
    if s >= g: return "🥇 GOLD"
    if s >= si: return "🥈 SILVER"
    if s >= b: return "🥉 BRONZE"
    return "🐣 no medal yet: keep digging"

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
print("Setup complete 🐾  | numpy", np.__version__, "| torch available:", HAS_TORCH)

---
# Part 1 · Introduction (Chapter 1)
### 📖 Recap
The book sorts machine learning into three families:

**Supervised learning** learns a mapping from an input $\mathbf{x}$ to an output $\mathbf{y}$ using example pairs $\{\mathbf{x}_i,\mathbf{y}_i\}$. The model is a *family* of equations $\mathbf{y}=\mathbf{f}[\mathbf{x},\boldsymbol\phi]$, and the parameters $\boldsymbol\phi$ pick one member of the family.
- **Regression**: continuous outputs (univariate or multivariate).
- **Classification**: discrete outputs (binary or multiclass).
- Inputs can be tabular, or *structured* (audio, images, text, molecules). Structured inputs are high-dimensional and their elements are related to one another.

**Unsupervised learning** learns from $\{\mathbf{x}_i\}$ alone, with no labels.
- **Generative models** learn to *synthesize* new examples. They can be probabilistic (giving $Pr(\mathbf{x})$) or not.
- **Latent variables** $\mathbf{z}$ capture the underlying structure, so that $\mathbf{x}$ can be generated from a simpler $\mathbf{z}$.

**Reinforcement learning**: an agent takes **actions** in an environment, changes its **state**, and receives **rewards**. The hard part is *temporal credit assignment*: working out which of many past actions earned a delayed reward.

Chapter 1 ends with **ethics**. Bias and fairness, explainability, weaponization, concentration of power and environmental cost all matter at TBio too. What data we train on shapes what the model "believes".

### 🧩 Puzzle 1.1 ⭐ Sort the field-station requests
The station manager emails you eight requests. Label each one with **one** of:
`"regression"`, `"multivariate regression"`, `"binary classification"`, `"multiclass classification"`, `"unsupervised"`, `"generative"`, `"reinforcement"`.

In [ ]:
requests = {
 "A": "Predict a pup's adult body mass (g) from its mass at 30 days",
 "B": "Decide whether a 2-second audio clip contains an alarm call (yes/no)",
 "C": "Identify which predator (eagle, jackal, cobra) an alarm call refers to",
 "D": "Group 500 unlabelled GPS foraging tracks into 'foraging styles'",
 "E": "Synthesize realistic new meerkat calls for a playback experiment",
 "F": "Train a robot sentinel that learns when to stand guard to maximise mob survival",
 "G": "Predict the mob's (latitude, longitude) at sunset",
 "H": "TBio cameo: predict whether a protein forms condensates at a given concentration",
}
my_answers = {
    "A": None, "B": None, "C": None, "D": None,   # TODO: replace None with a label string
    "E": None, "F": None, "G": None, "H": None,
}

In [ ]:
# 🧪 Check puzzle 1.1
def _t11():
    if any(v is None for v in my_answers.values()): raise NotImplementedError
    key = _K11
    n = sum(my_answers[k].strip().lower() == v for k, v in key.items())
    print(f"   {n}/8 correct")
    return n >= 7
_K11 = dict(A="regression", B="binary classification", C="multiclass classification",
            D="unsupervised", E="generative", F="reinforcement",
            G="multivariate regression", H="binary classification")
check("1.1", 1, _t11, "You know your way around the field station.",
      "Is the output a number, several numbers, a yes/no, one of many classes... or is there no label at all?")

In [ ]:
#@title 🔑 Solution 1.1 — double-click to peek (have a go first!)
peek('1.1')
my_answers = dict(A="regression", B="binary classification", C="multiclass classification",
                  D="unsupervised", E="generative", F="reinforcement",
                  G="multivariate regression", H="binary classification")
print(my_answers)

### 🧩 Puzzle 1.2 ⭐ Why structured inputs need structured models
A collar microphone records **10 seconds** of audio at **16 kHz**. We flatten that into a vector $\mathbf{x}$ and feed it into a *single* fully connected layer with **1,000** hidden units, each with its own bias.
How many parameters is that? Chapter 4 (§4.5.4) uses this kind of number to motivate convolutional networks.

In [ ]:
n_inputs = None        # TODO
n_params_layer = None  # TODO  (weights + biases)

In [ ]:
# 🧪 Check puzzle 1.2
def _t12():
    if n_inputs is None or n_params_layer is None: raise NotImplementedError
    return n_inputs == 160_000 and n_params_layer == 160_000 * 1000 + 1000
check("1.2", 1, _t12, "160 million parameters for ONE layer. This is why chapter 10 (convolutions) exists.",
      "Inputs = seconds × samples/second. Each hidden unit has one weight per input plus one bias.")

In [ ]:
#@title 🔑 Solution 1.2 — double-click to peek (have a go first!)
peek('1.2')
n_inputs = 10 * 16_000
n_params_layer = n_inputs * 1000 + 1000
print(n_inputs, f"{n_params_layer:,}")

---
# Part 2 · Supervised learning (Chapter 2): 🍼 the Great Pup Weigh-in
### 📖 Recap
Every morning the station weighs the pups on a kitchen scale (with a boiled-egg bribe). We want to predict **mass gained (g)** from **hours spent foraging** the day before.

**Model** (1D linear regression, eq 2.4):
$$y = \mathrm{f}[x,\boldsymbol\phi] = \phi_0 + \phi_1 x .$$

**Loss**: the least squares loss (eq 2.5) measures the mismatch over the training set:
$$L[\boldsymbol\phi] = \sum_{i=1}^{I}\big(\mathrm{f}[x_i,\boldsymbol\phi]-y_i\big)^2=\sum_{i=1}^{I}\big(\phi_0+\phi_1x_i-y_i\big)^2 .$$

**Training** means finding $\hat{\boldsymbol\phi}=\underset{\boldsymbol\phi}{\mathrm{argmin}}\, L[\boldsymbol\phi]$. **Testing** means measuring performance on *new* data to see how well the model **generalizes** (underfitting / overfitting; much more on this in Part 8).

In [ ]:
# 💻 The weigh-in data (synthetic, but the pups are very real in spirit)
rng = np.random.default_rng(2)
I = 20
hours = np.sort(rng.uniform(0.5, 7.5, I))                 # x: foraging hours
gain = 1.5 + 2.2 * hours + rng.normal(0, 2.0, I)          # y: grams gained (true phi = [1.5, 2.2])

def f_linear(x, phi0, phi1):
    return phi0 + phi1 * x

def loss_ls(phi0, phi1, x, y):
    return np.sum((f_linear(x, phi0, phi1) - y) ** 2)

plt.scatter(hours, gain, color=TB["orange"], zorder=3, label="pups")
xx = np.linspace(0, 8, 100)
plt.plot(xx, f_linear(xx, 0, 3), "--", color=TB["grey"], label=r"guess $\phi=[0,3]$")
plt.xlabel("foraging hours $x$"); plt.ylabel("mass gain $y$ (g)"); plt.legend(); plt.title("The Great Pup Weigh-in")
plt.show()
print("Loss of the guess:", round(loss_ls(0, 3, hours, gain), 2))

In [ ]:
# 💻 The loss is a SURFACE over parameter space (cf. Fig 2.3)
P0, P1 = np.meshgrid(np.linspace(-6, 10, 201), np.linspace(0, 4.5, 201))
Lgrid = ((P0[..., None] + P1[..., None] * hours - gain) ** 2).sum(-1)
fig, ax = plt.subplots(figsize=(6, 4.5))
cs = ax.contourf(P0, P1, np.log(Lgrid), 30, cmap="viridis")
ax.contour(P0, P1, np.log(Lgrid), 15, colors="w", linewidths=0.4)
plt.colorbar(cs, label="log loss"); ax.set_xlabel(r"intercept $\phi_0$"); ax.set_ylabel(r"slope $\phi_1$")
ax.set_title("Loss landscape: one valley, one minimum"); plt.show()

### 🧩 Puzzle 2.1 ⭐ Which way is downhill? (book P2.1)
Differentiate the loss. Write $r_i=\phi_0+\phi_1x_i-y_i$ for the residual of point $i$. Then
$$\frac{\partial L}{\partial \phi_0}=\;?\qquad\frac{\partial L}{\partial \phi_1}=\;?$$
Work it out on paper, then implement `grad_ls`. The checker compares your answer with a finite-difference estimate.

In [ ]:
def grad_ls(phi0, phi1, x, y):
    """Return (dL/dphi0, dL/dphi1) for the least squares loss."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 2.1
def _t21():
    for p in [(0., 3.), (1.5, 2.2), (-4, 0.7)]:
        g = grad_ls(*p, hours, gain)
        gn = num_grad(lambda q: loss_ls(q[0], q[1], hours, gain), p)
        if not close(g, gn, 1e-3): return False
    return True
check("2.1", 1, _t21, "The gradient of a sum is the sum of the gradients.",
      "Chain rule: d(r^2)/dphi = 2 r * dr/dphi, where dr/dphi0 = 1 and dr/dphi1 = x_i.")

In [ ]:
#@title 🔑 Solution 2.1 — double-click to peek (have a go first!)
peek('2.1')
def grad_ls(phi0, phi1, x, y):
    r = phi0 + phi1 * x - y
    return 2 * np.sum(r), 2 * np.sum(r * x)

### 🧩 Puzzle 2.2 ⭐⭐ Closed form (book P2.2)
Set both derivatives to zero:
$$\sum_i(\phi_0+\phi_1x_i-y_i)=0,\qquad \sum_i(\phi_0+\phi_1x_i-y_i)\,x_i=0 .$$
The first gives $\phi_0=\bar y-\phi_1\bar x$. Substitute it into the second and solve for $\phi_1$. You should find a ratio of a *covariance* to a *variance*. Implement `closed_form`.

In [ ]:
def closed_form(x, y):
    """Return (phi0_hat, phi1_hat) minimising the least squares loss."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 2.2
def _t22():
    p = closed_form(hours, gain)
    return close(p, np.polyfit(hours, gain, 1)[::-1], 1e-8)
check("2.2", 2, _t22, "Linear regression is convex with a closed form, but neural nets won't be this kind (chapter 6).",
      "phi1 = sum((x-xbar)(y-ybar)) / sum((x-xbar)^2)")

In [ ]:
#@title 🔑 Solution 2.2 — double-click to peek (have a go first!)
peek('2.2')
def closed_form(x, y):
    xb, yb = x.mean(), y.mean()
    phi1 = np.sum((x - xb) * (y - yb)) / np.sum((x - xb) ** 2)
    return yb - phi1 * xb, phi1
print(closed_form(hours, gain))

In [ ]:
# 💻 Gradient descent: "walk downhill" (Fig 2.4). Watch it zig-zag along a narrow valley.
def gradient_descent_lr(phi, x, y, lr=1e-3, n_steps=4000):
    path = [np.array(phi, float)]
    for _ in range(n_steps):
        g = np.array(grad_ls(*path[-1], x, y))
        path.append(path[-1] - lr * g)
    return np.array(path)

path = gradient_descent_lr([-5, 4.0], hours, gain)
best = closed_form(hours, gain)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].contourf(P0, P1, np.log(Lgrid), 30, cmap="viridis")
ax[0].plot(path[:, 0], path[:, 1], ".-", color=TB["orange"], ms=2, lw=1)
ax[0].plot(*best, "*", color="w", ms=14); ax[0].set_xlabel(r"$\phi_0$"); ax[0].set_ylabel(r"$\phi_1$")
ax[0].set_title("GD path in parameter space")
ax[1].scatter(hours, gain, color=TB["orange"], zorder=3)
for k, c in zip([0, 5, 50, 500, 4000], plt.cm.viridis(np.linspace(0, 0.9, 5))):
    ax[1].plot(xx, f_linear(xx, *path[k]), color=c, label=f"step {k}")
ax[1].legend(); ax[1].set_title("...and the corresponding lines"); plt.show()
print("GD after 4000 steps:", path[-1], " closed form:", np.array(best))

### 🧩 Puzzle 2.3 ⭐⭐⭐ Discriminative vs generative (book P2.3)
The station's statistician insists on a **generative** model: hours are "caused" by gain, so $x=\mathrm{g}[y,\boldsymbol\phi']=\phi'_0+\phi'_1y$, fitted by least squares on $x$. Inference then uses the **inverse**, $y=\mathrm{g}^{-1}[x,\boldsymbol\phi']=(x-\phi'_0)/\phi'_1$.

1. Fit $\boldsymbol\phi'$ (you can reuse `closed_form` with the arguments swapped).
2. Write the implied prediction line as $y = a + b x$ and return `(a, b)`.
3. Is it the same line as the discriminative fit? Look at the plot and work out why. (Hint: which errors does each loss penalise: vertical or horizontal?)

In [ ]:
def generative_line(x, y):
    """Fit x = p0 + p1*y by least squares, then return (a, b) of the inverse line y = a + b*x."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 2.3
def _t23():
    a, b = generative_line(hours, gain)
    p1 = np.polyfit(gain, hours, 1)
    return close((a, b), (-p1[1] / p1[0], 1 / p1[0]), 1e-8)
check("2.3", 3, _t23, "Different loss = different answer, even with the same family of lines.",
      "If x = p0 + p1*y, then y = -p0/p1 + (1/p1)*x.")

In [ ]:
#@title 🔑 Solution 2.3 — double-click to peek (have a go first!)
peek('2.3')
def generative_line(x, y):
    p0, p1 = closed_form(y, x)   # regress x on y
    return -p0 / p1, 1 / p1
a, b = generative_line(hours, gain); d0, d1 = closed_form(hours, gain)
plt.scatter(hours, gain, color=TB["orange"], zorder=3)
plt.plot(xx, d0 + d1 * xx, color=TB["green"], label=f"discriminative  y={d0:.2f}+{d1:.2f}x")
plt.plot(xx, a + b * xx, color=TB["purple"], label=f"generative (inverted)  y={a:.2f}+{b:.2f}x")
plt.legend(); plt.title("Same data, two different lines"); plt.show()
print("The generative fit minimises HORIZONTAL errors, so its inverted line is steeper.\n"
      "The two only agree when the points are perfectly collinear (correlation = ±1).")

In [ ]:
#@title 🎛️ Optional: beat the computer by hand (sliders; needs ipywidgets, which Colab has)
try:
    from ipywidgets import interact, FloatSlider
    def _play(phi0=0.0, phi1=3.0):
        plt.figure(figsize=(5, 3.2)); plt.scatter(hours, gain, color=TB["orange"])
        plt.plot(xx, f_linear(xx, phi0, phi1), color=TB["green"])
        L = loss_ls(phi0, phi1, hours, gain); Lb = loss_ls(*closed_form(hours, gain), hours, gain)
        plt.title(f"your loss {L:.1f}   vs   optimum {Lb:.1f}"); plt.show()
    interact(_play, phi0=FloatSlider(min=-6, max=10, step=0.1, value=0),
             phi1=FloatSlider(min=0, max=4.5, step=0.05, value=3))
except Exception as e:
    print("Sliders unavailable here:", e)

---
# Part 3 · Shallow neural networks (Chapter 3): ⛏️ digging burrows with ReLUs
### 📖 Recap
A shallow network with one input, one output and three hidden units has **ten parameters** (eq 3.1):
$$y=\phi_0+\phi_1\,a[\theta_{10}+\theta_{11}x]+\phi_2\,a[\theta_{20}+\theta_{21}x]+\phi_3\,a[\theta_{30}+\theta_{31}x],$$
with the **ReLU** activation $a[z]=\mathrm{ReLU}[z]=\max(0,z)$ (eq 3.2).

It works in three stages:
1. Compute three **linear functions** of the input; these are the *pre-activations*.
2. **Clip** each one at zero with the ReLU to get the hidden units $h_d$ (the *activations*).
3. Take a **weighted sum** of the hidden units and add an offset $\phi_0$.

The result is a **continuous piecewise-linear** function. Each hidden unit contributes one "joint", at $x=-\theta_{d0}/\theta_{d1}$, so $D$ hidden units give at most $D+1$ linear regions. Each region corresponds to one **activation pattern** (which units are active and which are clipped).

**Universal approximation theorem:** with enough hidden units, a shallow net can approximate any continuous function on a compact domain to arbitrary precision.

In [ ]:
# 💻 Build the 10-parameter network. These are the book's own parameters (P3.8 / Fig 3.3).
def relu(z):
    return np.maximum(z, 0.0)

def shallow_1_1_3(x, act, phi, theta):
    """phi = [phi0, phi1, phi2, phi3]; theta = 3x2 array [[th10, th11], [th20, th21], [th30, th31]].
    Returns output y plus the intermediate pre-activations and activations (for plotting)."""
    pre = theta[:, [0]] + theta[:, [1]] * x[None, :]     # (3, N) linear functions
    h = act(pre)                                         # (3, N) hidden units
    y = phi[0] + phi[1:] @ h                             # weighted sum + offset
    return y, pre, h

PHI_BOOK = np.array([-0.23, -1.3, 1.3, 0.66])
THETA_BOOK = np.array([[-0.2, 0.4], [-0.9, 0.9], [1.1, -0.7]])

def plot_decomposition(act, phi, theta, title=""):
    x = np.linspace(0, 2, 400); y, pre, h = shallow_1_1_3(x, act, phi, theta)
    fig, ax = plt.subplots(3, 4, figsize=(12, 6.5), sharex=True)
    for d in range(3):
        ax[d, 0].plot(x, pre[d], color=TB["blue"]); ax[d, 0].set_title(fr"$\theta_{{{d+1}0}}+\theta_{{{d+1}1}}x$")
        ax[d, 1].plot(x, h[d], color=TB["green"]); ax[d, 1].set_title(fr"$h_{d+1}$")
        ax[d, 2].plot(x, phi[d + 1] * h[d], color=TB["orange"]); ax[d, 2].set_title(fr"$\phi_{d+1}h_{d+1}$")
        for c in range(3): ax[d, c].set_ylim(-1, 1)
    for d in range(3): ax[d, 3].axis("off")
    big = fig.add_subplot(1, 4, 4); big.plot(x, y, color="k", lw=2)
    big.set_title(r"$y=\phi_0+\sum_d\phi_dh_d$"); big.set_ylim(-1, 1)
    fig.suptitle(title or "Fig 3.3 recreated", fontweight="bold"); plt.tight_layout(); plt.show()

plot_decomposition(relu, PHI_BOOK, THETA_BOOK)

### 🧩 Puzzle 3.1 ⭐ Where are the joints? What are the slopes? (book P3.3)
- Hidden unit $d$ switches on or off where $\theta_{d0}+\theta_{d1}x=0$, i.e. at $x=-\theta_{d0}/\theta_{d1}$.
- Inside a region, the slope is $\sum_{d\,\in\,\text{active}}\phi_d\,\theta_{d1}$.

Implement both functions. `region_slopes` must return the slopes from **left to right** over the whole real line: four values for three joints.

In [ ]:
def joints(theta):
    """Sorted x-positions of the joints."""
    # TODO
    raise NotImplementedError

def region_slopes(phi, theta):
    """Slopes of the len(theta)+1 linear regions, left to right."""
    # TODO  (tip: pick a test point inside each region and ask which units are active)
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 3.1
def _t31():
    j = joints(THETA_BOOK); s = region_slopes(PHI_BOOK, THETA_BOOK)
    x = np.array([j[0] - 1, (j[0] + j[1]) / 2, (j[1] + j[2]) / 2, j[2] + 1]); e = 1e-5
    num = (shallow_1_1_3(x + e, relu, PHI_BOOK, THETA_BOOK)[0] - shallow_1_1_3(x - e, relu, PHI_BOOK, THETA_BOOK)[0]) / (2 * e)
    print("   joints:", np.round(j, 4), " slopes:", np.round(s, 4))
    return close(j, [0.5, 1.0, 1.1 / 0.7], 1e-6) and close(s, num, 1e-5)
check("3.1", 1, _t31, "You can read a ReLU net like a map.",
      "joints = -theta[:,0]/theta[:,1]; a unit is active where theta0 + theta1*x > 0.")

In [ ]:
#@title 🔑 Solution 3.1 — double-click to peek (have a go first!)
peek('3.1')
def joints(theta):
    return np.sort(-theta[:, 0] / theta[:, 1])

def region_slopes(phi, theta):
    j = joints(theta)
    tests = np.concatenate([[j[0] - 1], (j[:-1] + j[1:]) / 2, [j[-1] + 1]])
    slopes = []
    for x in tests:
        active = theta[:, 0] + theta[:, 1] * x > 0
        slopes.append(np.sum(phi[1:][active] * theta[active, 1]))
    return np.array(slopes)
print(joints(THETA_BOOK), region_slopes(PHI_BOOK, THETA_BOOK))

### 🧩 Puzzle 3.2 ⭐⭐ Slopes aren't independent (book P3.9 + P3.2)
With 3 units there are four regions but only three independent slopes. Show that **slope(region 3) = slope(region 1) + slope(region 4)** for the Fig 3.3 network.

Then record the **activation pattern** of each region (which units are on), as a list of four sets using unit numbers 1–3.

*Why does the identity hold? The three units' "switch-on" directions differ: units 1 and 2 have positive $\theta_{d1}$ and switch on to the right, while unit 3 has negative $\theta_{31}$ and switches on to the left.*

In [ ]:
activation_patterns = [None, None, None, None]   # TODO e.g. [{3}, {...}, {...}, {...}]

In [ ]:
# 🧪 Check puzzle 3.2
def _t32():
    if any(p is None for p in activation_patterns): raise NotImplementedError
    s = region_slopes(PHI_BOOK, THETA_BOOK)
    ok_id = np.isclose(s[2], s[0] + s[3])
    return ok_id and [set(p) for p in activation_patterns] == [{3}, {1, 3}, {1, 2, 3}, {1, 2}]
check("3.2", 2, _t32, "Region 3 has all three units on; regions 1 and 4 split them between them.",
      "Plug test points into theta0 + theta1*x > 0 for each unit.")

In [ ]:
#@title 🔑 Solution 3.2 — double-click to peek (have a go first!)
peek('3.2')
activation_patterns = [{3}, {1, 3}, {1, 2, 3}, {1, 2}]
s = region_slopes(PHI_BOOK, THETA_BOOK)
print("slopes", s, "| s3 == s1 + s4 ?", np.isclose(s[2], s[0] + s[3]))
# Proof: s1 = phi3*th31 (only unit 3), s4 = phi1*th11 + phi2*th21 (units 1,2), s3 = all three = s1 + s4.

### 🧩 Puzzle 3.3 ⭐⭐ Non-negative homogeneity (book P3.5 & P3.6)
For $\alpha>0$, $\mathrm{ReLU}[\alpha z]=\alpha\,\mathrm{ReLU}[z]$. So if we multiply $\theta_{d0},\theta_{d1}$ by $\alpha$ and divide $\phi_d$ by $\alpha$, the function shouldn't change. Implement `rescale_unit`, then try $\alpha=-1$ and see what breaks.

*Consequence: the loss surface has whole curves of equivalent minima, so the least squares minimum is **not unique** (book P3.7).*

In [ ]:
def rescale_unit(phi, theta, d, alpha):
    """Return new (phi, theta) with hidden unit d (0-based) rescaled by alpha (input side) and 1/alpha (output side)."""
    # TODO (don't modify the inputs in place!)
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 3.3
def _t33():
    x = np.linspace(-3, 3, 301); y0 = shallow_1_1_3(x, relu, PHI_BOOK, THETA_BOOK)[0]
    p2, t2 = rescale_unit(PHI_BOOK, THETA_BOOK, 1, 3.7)
    same = close(shallow_1_1_3(x, relu, p2, t2)[0], y0, 1e-9)
    p3, t3 = rescale_unit(PHI_BOOK, THETA_BOOK, 1, -1.0)
    differs = not close(shallow_1_1_3(x, relu, p3, t3)[0], y0, 1e-3)
    untouched = close(THETA_BOOK, [[-0.2, 0.4], [-0.9, 0.9], [1.1, -0.7]])
    return same and differs and untouched
check("3.3", 2, _t33, "With alpha < 0 the unit flips which side of its joint is active: a different function.",
      "copy first: phi = phi.copy(); theta = theta.copy()")

In [ ]:
#@title 🔑 Solution 3.3 — double-click to peek (have a go first!)
peek('3.3')
def rescale_unit(phi, theta, d, alpha):
    phi, theta = phi.copy(), theta.copy()
    theta[d] *= alpha; phi[d + 1] /= alpha
    return phi, theta

In [ ]:
# 💻 The activation-function zoo (Fig 3.13)
from math import erf
zoo = {
    "ReLU": relu,
    "leaky ReLU (0.1)": lambda z: np.where(z > 0, z, 0.1 * z),
    "sigmoid": lambda z: 1 / (1 + np.exp(-z)),
    "tanh": np.tanh,
    "softplus": lambda z: np.log1p(np.exp(z)),
    "GELU": lambda z: 0.5 * z * (1 + np.vectorize(erf)(z / np.sqrt(2))),
    "Swish (β=1)": lambda z: z / (1 + np.exp(-z)),
    "HardSwish": lambda z: np.where(z < -3, 0, np.where(z > 3, z, z * (z + 3) / 6)),
}
z = np.linspace(-4, 4, 400)
fig, axs = plt.subplots(2, 4, figsize=(12, 5), sharex=True)
for ax, (name, fn) in zip(axs.ravel(), zoo.items()):
    ax.plot(z, fn(z), color=TB["green"], lw=2); ax.set_title(name); ax.axhline(0, color="k", lw=0.5)
plt.tight_layout(); plt.show()

### 🧩 Puzzle 3.4 ⭐ Swap the activation (book P3.8)
Re-run `plot_decomposition` with the three activations below, then match each one to the family of functions it produces.

$$\text{heaviside}[z]=\begin{cases}0&z<0\\1&z\ge0\end{cases},\qquad \text{rect}[z]=\begin{cases}0&z<0\\1&0\le z\le1\\0&z>1\end{cases},\qquad\tanh[z].$$

Families:
- **(a)** smooth sums of S-shaped curves
- **(b)** piecewise-constant staircases with up to 3 jumps
- **(c)** sums of up to 3 rectangular "boxes" (up to 6 jumps)

In [ ]:
heaviside = lambda z: (z >= 0).astype(float)
rect = lambda z: ((z >= 0) & (z <= 1)).astype(float)
for nm, fn in [("heaviside", heaviside), ("rect", rect), ("tanh", np.tanh)]:
    plot_decomposition(fn, PHI_BOOK, THETA_BOOK, title=f"activation = {nm}")

family = {"heaviside": None, "rect": None, "tanh": None}   # TODO: "a", "b" or "c"

In [ ]:
# 🧪 Check puzzle 3.4
def _t34():
    if None in family.values(): raise NotImplementedError
    return family == {"heaviside": "b", "rect": "c", "tanh": "a"}
check("3.4", 1, _t34, "Heaviside nets were the original perceptrons, and you can't train them by gradient descent. Why not?",
      "Look at the right-hand column of each figure.")

In [ ]:
#@title 🔑 Solution 3.4 — double-click to peek (have a go first!)
peek('3.4')
family = {"heaviside": "b", "rect": "c", "tanh": "a"}

### 🧩 Puzzle 3.5 ⭐⭐ Universal approximation, by construction
The **sentinel vigilance curve** $v(t)$ gives how alert the mob's sentinel is through the day ($t\in[0,1]$).

Build a shallow net that **interpolates** $v$ at $D+1$ knots $t_0<\dots<t_D$. Use hidden units $h_d=\mathrm{ReLU}[t-t_d]$ for $d=0,\dots,D-1$ (so $\theta_{d0}=-t_d$, $\theta_{d1}=1$), set $\phi_0=v(t_0)$, and choose $\phi_d$ so that the slope *changes* by the right amount at each knot:
$$\phi_{d}= s_d - s_{d-1},\quad s_d=\frac{v(t_{d+1})-v(t_d)}{t_{d+1}-t_d},\quad s_{-1}=0 .$$
This is the argument behind Fig 3.5: more units, more regions, smaller error.

In [ ]:
def vigilance(t):
    return 0.6 + 0.3 * np.sin(2 * np.pi * t) * np.exp(-1.5 * t) + 0.15 * np.cos(9 * t)

def shallow_1d(x, phi0, phis, theta0s, theta1s):
    """General 1-input/1-output shallow ReLU net with D units."""
    return phi0 + relu(theta0s[:, None] + theta1s[:, None] * x[None, :]).T @ phis

In [ ]:
def make_interpolating_net(tk, vk):
    """Knots tk (D+1,) and values vk (D+1,) -> (phi0, phis, theta0s, theta1s) with D hidden units."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 3.5
def _t35():
    tk = np.linspace(0, 1, 51); net = make_interpolating_net(tk, vigilance(tk))
    t = np.linspace(0, 1, 2001)
    err = np.max(np.abs(shallow_1d(t, *net) - vigilance(t)))
    knots = close(shallow_1d(tk, *net), vigilance(tk), 1e-9)
    print(f"   max error with 50 units: {err:.4f}")
    return knots and err < 0.01
check("3.5", 2, _t35, "Universal approximation, demonstrated by construction.",
      "slopes s = np.diff(vk)/np.diff(tk); phis = np.diff(np.concatenate([[0], s]))")

In [ ]:
#@title 🔑 Solution 3.5 — double-click to peek (have a go first!)
peek('3.5')
def make_interpolating_net(tk, vk):
    s = np.diff(vk) / np.diff(tk)
    phis = np.diff(np.concatenate([[0.0], s]))
    return vk[0], phis, -tk[:-1], np.ones(len(tk) - 1)

In [ ]:
# 💻 More units -> more regions -> smaller error
t = np.linspace(0, 1, 1000)
fig, axs = plt.subplots(1, 4, figsize=(13, 3), sharey=True)
for ax, D in zip(axs, [3, 6, 12, 48]):
    tk = np.linspace(0, 1, D + 1); y = shallow_1d(t, *make_interpolating_net(tk, vigilance(tk)))
    ax.plot(t, vigilance(t), "--", color=TB["grey"]); ax.plot(t, y, color=TB["green"], lw=2)
    ax.set_title(f"D={D}  max err={np.max(np.abs(y - vigilance(t))):.3f}")
axs[0].set_ylabel("vigilance"); plt.tight_layout(); plt.show()

### 📖 Multivariate inputs: regions become convex polygons
With $D_i=2$ inputs, each hidden unit's joint becomes a **line** across the input plane, and the regions become **convex polygons** (Fig 3.8). **Zaslavsky (1975):** $D$ hyperplanes cut $\mathbb{R}^{D_i}$ into at most
$$\sum_{j=0}^{D_i}\binom{D}{j}$$
regions. Below we colour each pixel of the plane by its activation pattern and count the distinct colours.

In [ ]:
def count_regions_2d(Om, b, lim=4, n=600):
    g = np.linspace(-lim, lim, n); X1, X2 = np.meshgrid(g, g)
    pre = b[:, None] + Om @ np.stack([X1.ravel(), X2.ravel()])
    codes = (pre > 0).astype(int).T @ (2 ** np.arange(len(b)))
    return codes.reshape(n, n), len(np.unique(codes))

r = np.random.default_rng(5)
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for ax, D in zip(axs, [3, 5, 12]):
    img, n = count_regions_2d(r.normal(size=(D, 2)), r.normal(size=D))
    ax.imshow(img % 17, cmap="tab20", origin="lower", extent=[-4, 4, -4, 4])
    ax.set_title(f"D={D} hidden units → {n} regions"); ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$")
plt.tight_layout(); plt.show()

### 🧩 Puzzle 3.6 ⭐ Counting regions and parameters (book P3.17 & P3.18)
1. `max_regions(D, Di)`: Zaslavsky's bound. Note that the sum stops at $\min(D_i, D)$.
2. `n_params_shallow(Di, D, Do)`: a general shallow network (eqs 3.11–3.12) has how many weights and biases?

The checker uses the book's numbers: Fig 3.11 has twenty parameters, and $D=500$ units on a $D_i=100$-dimensional input give more than $10^{107}$ regions.

In [ ]:
def max_regions(D, Di):
    # TODO (math.comb is your friend)
    raise NotImplementedError

def n_params_shallow(Di, D, Do):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 3.6
def _t36():
    return (max_regions(3, 2) == 7 and max_regions(5, 2) == 16 and max_regions(4, 1) == 5
            and max_regions(3, 3) == 8 and max_regions(500, 100) > 10 ** 107
            and n_params_shallow(3, 3, 2) == 20 and n_params_shallow(1, 4, 2) == 18
            and n_params_shallow(2, 3, 1) == 13 and n_params_shallow(100, 500, 1) == 51001)
check("3.6", 1, _t36, "51,001 parameters can carve 10^107 regions. Small network, huge function space.",
      "Each hidden unit has Di weights + 1 bias; each output has D weights + 1 bias.")

In [ ]:
#@title 🔑 Solution 3.6 — double-click to peek (have a go first!)
peek('3.6')
def max_regions(D, Di):
    return sum(math.comb(D, j) for j in range(min(Di, D) + 1))

def n_params_shallow(Di, D, Do):
    return D * (Di + 1) + Do * (D + 1)

### 🏆 Challenge 3 ⭐⭐ Dig the burrow
The mob's main burrow has a known cross-section (the dashed line below). Hand-pick the ten parameters of a **3-unit** shallow net so that it matches the burrow profile on $x\in[0,1]$.
- 🥉 MSE < 0.01
- 🥈 MSE < 0.001
- 🥇 exact (MSE < 1e-10)

*Strategy: find the three joints and the slope changes at each joint, then use units of the form $\phi_d\,\mathrm{ReLU}[x-c_d]$.*

In [ ]:
def burrow(x):
    return np.select([x < 0.2, x < 0.5, x < 0.7], [0.0, -2 * (x - 0.2), -0.6 + 3 * (x - 0.5)], 0.0)
xb = np.linspace(0, 1, 501)
plt.figure(figsize=(6, 2.6)); plt.fill_between(xb, burrow(xb), -0.8, color=TB["sand"])
plt.plot(xb, burrow(xb), "k--"); plt.title("Burrow cross-section (sand in beige)"); plt.ylim(-0.8, 0.2); plt.show()

In [ ]:
my_phi = np.array([0.0, 0.0, 0.0, 0.0])                  # TODO [phi0, phi1, phi2, phi3]
my_theta = np.array([[0.0, 1.0], [0.0, 1.0], [0.0, 1.0]])  # TODO [[th10, th11], ...]

In [ ]:
# 🧪 Check puzzle C3
def _tc3():
    mse = np.mean((shallow_1_1_3(xb, relu, my_phi, my_theta)[0] - burrow(xb)) ** 2)
    print(f"   MSE = {mse:.2e}  →  {medal(mse, 1e-2, 1e-3, 1e-10)}")
    plt.figure(figsize=(6, 2.6)); plt.plot(xb, burrow(xb), "k--", label="burrow")
    plt.plot(xb, shallow_1_1_3(xb, relu, my_phi, my_theta)[0], color=TB["green"], lw=2, label="your net")
    plt.legend(); plt.show()
    return mse < 1e-10
check("C3", 2, _tc3, "Perfect burrow. The mob moves in tonight.",
      "Joints at 0.2, 0.5, 0.7. Slope goes 0 → -2 → +3 → 0, so the slope changes are -2, +5, -3.")

In [ ]:
#@title 🔑 Solution C3 — double-click to peek (have a go first!)
peek('C3')
my_phi = np.array([0.0, -2.0, 5.0, -3.0])
my_theta = np.array([[-0.2, 1.0], [-0.5, 1.0], [-0.7, 1.0]])

---
# Part 4 · Deep neural networks (Chapter 4): 🏜️ folding the dune sea
### 📖 Recap
**Composing** two shallow networks, $x\to y\to y'$, gives a deep network. The first network **folds** the input space onto itself, so several $x$ values map to the same $y$. The second network then applies its function, and that function is **replicated** at every fold (Figs 4.1 and 4.3).

Substituting the first network into the second gives a two-layer network (eqs 4.7–4.9):
$$\mathbf{h}=a[\boldsymbol\theta_0+\boldsymbol\theta x],\quad \mathbf{h}'=a[\boldsymbol\psi_0+\boldsymbol\Psi\mathbf{h}],\quad y'=\phi_0'+\boldsymbol\phi'\mathbf{h}' .$$
In the composed case $\boldsymbol\Psi$ is forced to be an outer product, so a general deep network is *more* expressive than the composition.

**General $K$-layer network** (eq 4.15):
$$\mathbf{h}_1=a[\boldsymbol\beta_0+\boldsymbol\Omega_0\mathbf{x}],\;\; \mathbf{h}_{k+1}=a[\boldsymbol\beta_k+\boldsymbol\Omega_k\mathbf{h}_k],\;\; \mathbf{y}=\boldsymbol\beta_K+\boldsymbol\Omega_K\mathbf{h}_K,$$
with $\boldsymbol\Omega_k\in\mathbb{R}^{D_{k+1}\times D_k}$ and $\boldsymbol\beta_k\in\mathbb{R}^{D_{k+1}}$.

**Shallow vs deep** (1 input, 1 output, $D>2$ units per layer):

| | parameters | max linear regions |
|---|---|---|
| shallow | $3D+1$ | $D+1$ |
| $K$ layers | $3D+1+(K-1)D(D+1)$ | $(D+1)^K$ |

In [ ]:
# 💻 Composition = folding.  Net 1 zig-zags [-1,1] -> [-1,1] with three alternating-slope pieces.
def net1(x):
    return shallow_1d(x, -1.0, np.array([3.0, -6.0, 6.0]), np.array([1.0, 1/3, -1/3]), np.ones(3))
def net2(y):   # joints at -0.3 and 0.5 (third joint at -2.0 is outside [-1, 1])
    return shallow_1d(y, -0.5, np.array([2.0, -3.0, 1.0]), np.array([-0.3, -0.5, 2.0]), np.array([-1.0, 1.0, 1.0]))

def count_regions_1d(fn, lo=-1, hi=1, n=400_001, tol=1e-6):
    """Count linear regions of a piecewise-linear fn on [lo, hi] by finding slope changes."""
    x = np.linspace(lo, hi, n); s = np.diff(fn(x)) / np.diff(x)
    flags = np.abs(np.diff(s)) > tol * (1 + np.abs(s).max())
    starts = np.flatnonzero(flags & ~np.concatenate([[False], flags[:-1]]))   # merge neighbours
    return len(starts) + 1

x = np.linspace(-1, 1, 2001)
fig, ax = plt.subplots(1, 3, figsize=(13, 3.6))
ax[0].plot(x, net1(x), color=TB["blue"], lw=2); ax[0].axhline(0.4, color=TB["orange"], ls=":")
ax[0].set_title(f"net 1 (folds): {count_regions_1d(net1)} regions"); ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[1].plot(x, net2(x), color=TB["purple"], lw=2); ax[1].set_title(f"net 2: {count_regions_1d(net2)} regions on [-1,1]")
ax[1].set_xlabel("y"); ax[1].set_ylabel("y'")
ax[2].text(0.5, 0.5, "?", fontsize=60, ha="center", va="center", color=TB["grey"], transform=ax[2].transAxes)
ax[2].set_title("net2(net1(x)) = ?"); plt.tight_layout(); plt.show()
print("(the dotted line in panel 1: three different x's land on the same y, so net 2 sees them as one point)")

In [ ]:
# 🧩 Puzzle 4.1 ⭐  Predict BEFORE you run the next line:
#    how many linear regions does net2(net1(x)) have on x in [-1, 1]?
my_prediction = None  # TODO

In [ ]:
# 🧪 Check puzzle 4.1
def _t41():
    if my_prediction is None: raise NotImplementedError
    truth = count_regions_1d(lambda x: net2(net1(x)))
    plt.figure(figsize=(4.5, 3.2)); plt.plot(x, net2(net1(x)), color=TB["green"], lw=2)
    plt.title(f"composition: {truth} regions"); plt.xlabel("x"); plt.ylabel("y'"); plt.show()
    return my_prediction == truth
check("4.1", 1, _t41, "3 folds × 3 regions = 9. Net 2's function gets copied (and flipped) onto each fold.",
      "Net 1 maps each of its 3 pieces onto the WHOLE of [-1, 1].")

In [ ]:
#@title 🔑 Solution 4.1 — double-click to peek (have a go first!)
peek('4.1')
my_prediction = 9

### 🧩 Puzzle 4.2 ⭐⭐ The general deep network in matrix form (eq 4.15)
Implement `deep_forward` for any depth. `x` has shape `(Di, N)`, so the columns are examples. Return the output **and** lists of the pre-activations $\mathbf{f}_k$ and activations $\mathbf{h}_k$; we'll need them for backprop in Part 7.

Conventions (match the book's eq 7.24):
$\mathbf{f}_0=\boldsymbol\beta_0+\boldsymbol\Omega_0\mathbf{x}$, $\;\mathbf{h}_k=a[\mathbf{f}_{k-1}]$, $\;\mathbf{f}_k=\boldsymbol\beta_k+\boldsymbol\Omega_k\mathbf{h}_k$, and the output is $\mathbf{f}_K$.

In [ ]:
# 💻 Random initialisation for any layer sizes (we'll revisit the variance choice in Part 7)
def init_params(dims, sigma2=None, rng=np.random.default_rng(0)):
    """dims = [Di, D1, ..., DK, Do]. Normal weights (He variance 2/D_in by default), zero biases."""
    betas, omegas = [], []
    for d_in, d_out in zip(dims[:-1], dims[1:]):
        s2 = 2.0 / d_in if sigma2 is None else sigma2
        omegas.append(rng.normal(0, np.sqrt(s2), (d_out, d_in))); betas.append(np.zeros((d_out, 1)))
    return betas, omegas

In [ ]:
def deep_forward(x, betas, omegas, act=relu):
    """betas[k]: (D_{k+1},1), omegas[k]: (D_{k+1}, D_k). Returns (out, all_f, all_h)
    where all_f = [f_0, ..., f_K] and all_h = [x, h_1, ..., h_K] (all_h[0] is the input)."""
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 4.2
def _t42():
    rr = np.random.default_rng(1)
    th = rr.normal(size=(3, 2)); Ps = rr.normal(size=(3, 4)); ph = rr.normal(size=4)
    betas = [th[:, [0]], Ps[:, [0]], np.array([[ph[0]]])]; omegas = [th[:, [1]], Ps[:, 1:], ph[None, 1:]]
    xx = np.linspace(-2, 2, 50)
    # eq 4.10, written out by hand
    h = relu(th[:, [0]] + th[:, [1]] * xx); hp = relu(Ps[:, [0]] + Ps[:, 1:] @ h); ref = ph[0] + ph[1:] @ hp
    out, fs, hs = deep_forward(xx[None, :], betas, omegas)
    b2, o2 = init_params([3, 4, 2, 3, 2])
    out2 = deep_forward(np.ones((3, 5)), b2, o2)[0]
    return close(out[0], ref, 1e-10) and len(fs) == 3 and len(hs) == 3 and out2.shape == (2, 5)
check("4.2", 2, _t42, "One function for every fully connected net in this notebook.",
      "loop: f = beta + Omega @ h; then h = act(f) for every layer except the last.")

In [ ]:
#@title 🔑 Solution 4.2 — double-click to peek (have a go first!)
peek('4.2')
def deep_forward(x, betas, omegas, act=relu):
    all_f, all_h = [], [x]
    K = len(omegas) - 1
    for k in range(K + 1):
        f = betas[k] + omegas[k] @ all_h[-1]
        all_f.append(f)
        if k < K:
            all_h.append(act(f))
    return all_f[-1], all_f, all_h

### 🧩 Puzzle 4.3 ⭐ Matrix-shape bingo (Fig 4.6)
A network has $D_i=3$ inputs, three hidden layers of sizes $D_1=4$, $D_2=2$, $D_3=3$, and $D_o=2$ outputs. Fill in the shape of every weight matrix and bias vector as a tuple. Write biases as 1-tuples, e.g. `(4,)`.

In [ ]:
shapes = {"Omega0": None, "beta0": None, "Omega1": None, "beta1": None,
          "Omega2": None, "beta2": None, "Omega3": None, "beta3": None}   # TODO

In [ ]:
# 🧪 Check puzzle 4.3
def _t43():
    if None in shapes.values(): raise NotImplementedError
    ans = {"Omega0": (4, 3), "beta0": (4,), "Omega1": (2, 4), "beta1": (2,),
           "Omega2": (3, 2), "beta2": (3,), "Omega3": (2, 3), "beta3": (2,)}
    wrong = [k for k in ans if tuple(shapes[k]) != ans[k]]
    if wrong: print("   check:", wrong)
    return not wrong
check("4.3", 1, _t43, "Omega_k is (size of layer it feeds) x (size of layer it reads).",
      "Omega_k has shape (D_{k+1}, D_k); beta_k has length D_{k+1}.")

In [ ]:
#@title 🔑 Solution 4.3 — double-click to peek (have a go first!)
peek('4.3')
shapes = {"Omega0": (4, 3), "beta0": (4,), "Omega1": (2, 4), "beta1": (2,),
          "Omega2": (3, 2), "beta2": (3,), "Omega3": (2, 3), "beta3": (2,)}

### 🧩 Puzzle 4.4 ⭐⭐ Regions per parameter (§4.5.2, Fig 4.7a)
Implement the two formulas from the table at the top of this part. Then the cell after the check plots *max regions vs parameters* for $K=1\dots5$. The book's highlighted point is $K=5$, $D=10$: **471 parameters, 161,051 regions**.

In [ ]:
def n_params_deep_1d(D, K):
    # TODO
    raise NotImplementedError

def max_regions_deep_1d(D, K):
    # TODO
    raise NotImplementedError

In [ ]:
# 🧪 Check puzzle 4.4
def _t44():
    return (n_params_deep_1d(10, 5) == 471 and max_regions_deep_1d(10, 5) == 161051
            and n_params_deep_1d(7, 1) == 22 and max_regions_deep_1d(7, 1) == 8)
check("4.4", 2, _t44, "Depth buys regions much more cheaply than width does.",
      "Each extra layer adds D*(D+1) parameters and multiplies the region count by (D+1).")

In [ ]:
#@title 🔑 Solution 4.4 — double-click to peek (have a go first!)
peek('4.4')
def n_params_deep_1d(D, K):
    return 3 * D + 1 + (K - 1) * D * (D + 1)
def max_regions_deep_1d(D, K):
    return (D + 1) ** K

In [ ]:
plt.figure(figsize=(6, 4))
for K in range(1, 6):
    Ds = np.arange(3, 60)
    plt.loglog([n_params_deep_1d(D, K) for D in Ds], [max_regions_deep_1d(D, K) for D in Ds], label=f"K={K}")
plt.plot(471, 161051, "o", color="k", ms=8, mfc="none"); plt.annotate("K=5, D=10", (471, 161051), xytext=(600, 3e3))
plt.xlabel("number of parameters"); plt.ylabel("max linear regions"); plt.legend(); plt.title("Fig 4.7a")
plt.show()

### 🏆 Challenge 4 ⭐⭐⭐ The dune sea: depth efficiency (§4.5.3)
A wind-swept dune field is a **sawtooth**. The **tent layer**
$$t(x)=2\,\mathrm{ReLU}[x]-4\,\mathrm{ReLU}[x-\tfrac12]$$
uses only **2 hidden units** and maps $[0,1]$ onto $[0,1]$ with one fold. Composing it $K$ times gives $2^{K-1}$ dunes, i.e. $2^K$ linear regions, from just $2K$ hidden units.

1. Implement `dune_sea(x, K)` (it applies `tent` K times).
2. A *shallow* net needs one hidden unit per joint. How many hidden units would it need to reproduce `dune_sea(x, 10)` exactly on $[0,1]$?

In [ ]:
def tent(x):
    return 2 * relu(x) - 4 * relu(x - 0.5)

def dune_sea(x, K):
    # TODO
    raise NotImplementedError

shallow_units_needed_K10 = None   # TODO

In [ ]:
# 🧪 Check puzzle C4
def _tc4():
    if shallow_units_needed_K10 is None: raise NotImplementedError
    counts = [count_regions_1d(lambda x: dune_sea(x, K), 0, 1, n=200_001) for K in range(1, 7)]
    print("   regions for K=1..6:", counts)
    xs = np.linspace(0, 1, 2000)
    fig, axs = plt.subplots(1, 3, figsize=(12, 2.6), sharey=True)
    for a, K in zip(axs, [1, 3, 6]):
        a.fill_between(xs, dune_sea(xs, K), color=TB["sand"]); a.plot(xs, dune_sea(xs, K), color=TB["orange"])
        a.set_title(f"K={K}: {2*K} hidden units, {2**K} regions")
    plt.show()
    return counts == [2 ** K for K in range(1, 7)] and shallow_units_needed_K10 == 2 ** 10 - 1
check("C4", 3, _tc4, "20 deep units vs 1023 shallow ones: depth efficiency in one picture.",
      "2^K regions means 2^K - 1 joints, and a shallow net needs one unit per joint.")

In [ ]:
#@title 🔑 Solution C4 — double-click to peek (have a go first!)
peek('C4')
def dune_sea(x, K):
    for _ in range(K):
        x = tent(x)
    return x
shallow_units_needed_K10 = 2 ** 10 - 1

In [ ]:
# 💻 Reality check: RANDOM nets come nowhere near the maximum region count.
rr = np.random.default_rng(3); res = {}
for K, D in [(1, 30), (2, 6), (4, 3)]:
    cnts = []
    for _ in range(20):
        b, o = init_params([1] + [D] * K + [1], rng=rr)
        b = [bi + rr.normal(0, 0.5, bi.shape) for bi in b]
        cnts.append(count_regions_1d(lambda x: deep_forward(x[None, :], b, o)[0][0], -3, 3, n=100_001))
    res[(K, D)] = (n_params_deep_1d(D, K), np.mean(cnts), max_regions_deep_1d(D, K))
for (K, D), (p, m, mx) in res.items():
    print(f"K={K}, D={D:2d}: {p:3d} params | random init: {m:5.1f} regions on [-3,3] | theoretical max {mx}")
print("\nThe book's caveat (§4.5.2): many regions are possible, but they come with complex dependencies and symmetries.")

---
# 🏁 Your progress

In [ ]:
progress()

### ⏭️ Next time
**Session 2 · Losses and training (Ch 5–6):** where do loss functions come from, and how do we walk downhill on a non-convex surface?

*Built for the Transition Bio ML reading group · based on Prince, Understanding Deep Learning (MIT Press, CC-BY-NC-ND) · all data synthetic.*